# Spark Gold Processing

This notebook is run headlessly by Airflow after Silver data is ready.
It reads the Silver Iceberg facts and creates aggregated Gold tables.

**Tables produced:**
- `lake.analytics.player_season_stats` (Gold aggregate)

In [ ]:
# Initialize the Spark session with AQE optimizations
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# The Spark session uses config from spark-defaults.conf (Iceberg + MinIO)
spark = (SparkSession.builder
    .appName("BrasileiraoGoldProcessing")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
    .getOrCreate())

# Reduce log noise during headless execution
spark.sparkContext.setLogLevel("WARN")
print("Spark session created.")

In [ ]:
import os

# Season and league are injected by the Airflow DAG:
#   docker exec -e SEASON=<year> -e LEAGUE_KEY=<key>
# Falls back to sensible defaults for local/manual runs.
SEASON = int(os.getenv("SEASON", "2024"))
LEAGUE_KEY = os.getenv("LEAGUE_KEY", "BRA-Brasileirao")

# Read only the relevant (season, league) partition from Silver
df_player_match = (
    spark.read.table("lake.analytics.player_match_stats")
    .filter((F.col("season") == SEASON) & (F.col("league") == LEAGUE_KEY))
)

numeric_cols = [
    "total_goals", "goal_assists", "shots_on_target", "total_shots",
    "fouls_committed", "fouls_suffered", "yellow_cards", "red_cards",
    "own_goals", "offsides", "appearances", "saves", "shots_faced",
    "goals_conceded"
]
print(f"League:     {LEAGUE_KEY}")
print(f"Season:     {SEASON}")
print(f"Loaded Silver (season={SEASON}, league={LEAGUE_KEY}): {df_player_match.count()} rows")

In [ ]:
# ==========================================================================
# GOLD: player_season_stats
# Aggregated season-level player statistics (SUM across all matches).
# ==========================================================================

# Aggregation columns: sum all numeric stats across the season
agg_exprs = []
for col_name in numeric_cols:
    if col_name in df_player_match.columns:
        agg_exprs.append(F.sum(col_name).alias(col_name))

df_season_stats = (
    df_player_match
    .groupBy("player", "team", "league", "season")
    .agg(
        F.countDistinct("game").alias("matches_played"),
        # Sum of matches started
        F.sum(F.when(F.col("starter") == True, 1).otherwise(0)).alias("matches_started"),
        # All numeric stat aggregations
        *agg_exprs,
    )
    # Rename total_goals -> goals for clarity at season level
    .withColumnRenamed("total_goals", "goals")
    .withColumnRenamed("goal_assists", "assists")
    # Add goals+assists combined metric
    .withColumn("goal_contributions", F.col("goals") + F.col("assists"))
    # Add goals per match ratio
    .withColumn("goals_per_match", F.round(F.col("goals") / F.col("matches_played"), 2))
    .withColumn("processed_at", F.current_timestamp())
    .orderBy(F.desc("goals"))
)

print(f"Player season stats (Gold): {df_season_stats.count()} rows")
print("\nTop 10 scorers:")
df_season_stats.select(
    "player", "team", "matches_played", "goals", "assists",
    "goal_contributions", "goals_per_match"
).show(10, truncate=False)

In [ ]:
# ==========================================================================
# QUALITY GATES — fail fast before writing Gold table
# ==========================================================================

cnt = df_season_stats.count()
if cnt == 0:
    raise RuntimeError(
        f"Quality gate FAILED: player_season_stats for season={SEASON} is empty. "
        "Check that Silver processing completed correctly."
    )

# Every row must have a player name and team
for col_chk in ("player", "team"):
    nulls = df_season_stats.filter(F.col(col_chk).isNull() | (F.col(col_chk) == "")).count()
    if nulls > 0:
        raise RuntimeError(
            f"Quality gate FAILED: column '{col_chk}' has {nulls} null values in player_season_stats."
        )

print(f"Quality gates PASSED: player_season_stats has {cnt} rows for season={SEASON}")

In [ ]:
# ==========================================================================
# WRITE: Persist Gold table to Iceberg (partitioned by season + league)
# ==========================================================================

spark.sql("CREATE NAMESPACE IF NOT EXISTS lake.analytics")


def _write_partitioned(df, table: str, season: int, league: str) -> None:
    """Idempotent (season, league) partition write.

    First run  → creates the Iceberg table partitioned by (season, league).
    Later runs → atomically overwrites ONLY the matching partition;
                 all other seasons and leagues remain untouched.
    """
    fq = f"lake.analytics.{table}"
    if spark.catalog.tableExists(fq):
        df.writeTo(fq).overwritePartitions()
        print(f"  {fq}: overwrote (season={season}, league={league}) partition")
    else:
        df.writeTo(fq).partitionedBy("season", "league").create()
        print(f"  {fq}: created (partitioned by season, league)")


_write_partitioned(df_season_stats, "player_season_stats", SEASON, LEAGUE_KEY)
print("\nGold table written successfully.")

In [ ]:
# Stop the Spark session to free all memory
spark.stop()
print("Spark session stopped. Memory released.")